In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from models.attention_unet_3d import AttentionUNet3D
from utils.unets_helper_functions import (
    set_seed,
    train_one_epoch,
    validate_one_epoch,
    save_checkpoint,
    PatchDataset,
    evaluate_full_volume
)


In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))

Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 80
PATCHES_PER_CASE = 6

train_dataset = PatchDataset(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        augment=False)

val_dataset = PatchDataset(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=2,    
    pin_memory=True
)

In [6]:
model = AttentionUNet3D(
    in_channels=1,
    out_channels=7,
    base_filters=16
).to(device)

print("Model initialized.")

Model initialized.


In [7]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights = torch.tensor([0.1, 1, 1, 1, 1, 1, 1]).to(device)
ce_loss = nn.CrossEntropyLoss(weight=weights)
scaler = torch.cuda.amp.GradScaler()

C:\Users\dhanu\AppData\Local\Temp\ipykernel_8936\1548250201.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [8]:
EPOCHS = 40

PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "attention_unet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        ce_loss,
        device
    )

    val_loss, val_dice = validate_one_epoch(
        model,
        val_loader,
        ce_loss,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch+1}.pth")
        )
        print("Checkpoint Saved")

100%|██████████| 99/99 [04:34<00:00,  2.77s/it]



Epoch 0
Train Loss: 2.8346
Val Loss: 2.6008
Per Class Dice: [2.42432162e-01 5.91081654e-03 4.91796294e-08 7.07735789e-03
 3.73778992e-03 1.83764201e-01]
Saved Best Model


100%|██████████| 99/99 [04:10<00:00,  2.53s/it]



Epoch 1
Train Loss: 2.5491
Val Loss: 2.4636
Per Class Dice: [2.90280402e-01 1.86149233e-02 4.59095763e-07 1.17944456e-02
 7.17534182e-03 2.78749436e-01]
Saved Best Model


100%|██████████| 99/99 [03:41<00:00,  2.24s/it]


...


Epoch 37
Train Loss: 0.9957
Val Loss: 1.0254
Per Class Dice: [0.61559095 0.57658315 0.55555567 0.77156627 0.43418022 0.58131144]


100%|██████████| 99/99 [04:02<00:00,  2.45s/it]



Epoch 38
Train Loss: 0.9845
Val Loss: 1.0361
Per Class Dice: [0.6137176  0.80674572 0.66666667 0.64306937 0.71762863 0.56444355]


100%|██████████| 99/99 [04:04<00:00,  2.47s/it]



Epoch 39
Train Loss: 0.9775
Val Loss: 0.9916
Per Class Dice: [0.82103864 0.89958535 0.55555556 0.63425321 0.61709103 0.52578725]
Saved Best Model
Checkpoint Saved


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AttentionUNet3D().to(device)

model_path = "../experiments/attention_unet_fold0/best_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\dhanu\AppData\Local\Temp\ipykernel_28492\3142351894.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


In [11]:
print("Training Complete")

Training Complete


In [6]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.8818314276833449), np.float64(0.7960853794009235), np.float64(3.752345201679755e-09), np.float64(0.709722115534045), np.float64(0.7790560741198876), np.float64(0.8711903700630957)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.8232048513069763), np.float64(0.6526627619654866), np.float64(3.4340659222731253e-09), np.float64(0.6981543252398691), np.float64(0.7576708861986476), np.float64(0.8358394848209485)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.8291557480263313), np.float64(0.7853061225837952), np.float64(3.409478338187936e-09), np.float64(0.6997645829454541), np.float64(0.8087727612477), np.float64(0.878043593851592)]
Unique pred labels: [0 1 2 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.8099049525290686), np.float64(0.7444335204762977), np.float64(4.4964028574

## Attention UNet 3D Patch-Based Validation Performance

The model was trained for 40 epochs using patch-based validation.  
Below are the final patch validation metrics obtained at the best validation loss epoch.

### Final Training Statistics (Epoch 39 - Best Model)

- Train Loss: 0.9775  
- Validation Loss: 0.9916  

### Patch-Based Per-Class Dice (Validation)

- Mandible: 0.8210  
- Brainstem: 0.8996  
- Spinal Cord: 0.5556  
- Parotid Left: 0.6343  
- Parotid Right: 0.6171  
- Oral Cavity: 0.5258  


## Observations

- Brainstem achieved very strong performance (~0.90 Dice).
- Mandible segmentation is robust (>0.82 Dice).
- Parotid glands show moderate performance (~0.61–0.63).
- Spinal Cord performance is moderate (~0.56).
- Oral Cavity segmentation remains challenging.

Overall, the Attention UNet demonstrates strong patch-based segmentation performance, particularly for Mandible and Brainstem.


# Evaluating on Full Volume

~ (patch = 80 , stride = 60)

| Organ ID | Organ (Based on Your Map) | Mean Dice   | Std Dev |
| -------- | ------------------------- | ----------- | ------- |
| 1        | Bone Mandible             | **0.8425**  | 0.0365  |
| 2        | Brainstem                 | **0.7655**  | 0.0532  |
| 3        | Spinal Cord               | **~0.0000** | ~0.0000 |
| 4        | Parotid Left              | **0.6701**  | 0.0585  |
| 5        | Parotid Right             | **0.7425**  | 0.0841  |
| 6        | Oral Cavity               | **0.7889**  | 0.0839  |


Full Volume Evaluation Summary

**Strong Organs**

- Mandible (0.84) → Excellent and very stable
- Brainstem (0.77) → Strong and consistent
- Oral Cavity (0.79) → Significantly improved compared to UNet++
- Parotid Right (0.74) → Good bilateral performance

**Moderate**
- Parotid Left (0.67)
- Slightly lower but still consistent across cases
- Failure Case

- Spinal Cord (~0.00)
The model failed to predict class 3 in full-volume inference.
This suggests:
Extreme class imbalance sensitivity
Thin elongated structure suppression during sliding-window blending
Attention mechanism possibly suppressing weak spinal features